<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_12_NREAsimov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 12 — Asimov construction with neural ratio estimation

A normalizing flow is **not required** for the finite-sample Asimov construction. We only need reference events, positive estimates of the component-to-reference ratios, and a normalization convention used consistently in the Asimov weights and the likelihood.

We use the same five-dimensional signal and background simulator, detector response, and frozen PRESEL selection as Exercise 5. This time, all reference events come from the simulator: no flow is trained, loaded, sampled, or evaluated.

The experiment separates three questions:

1. How much does finite Monte Carlo integration move the minimum of a conventional simulator-weighted Asimov scan?
2. Does same-reference-sample normalization remove that displacement exactly in the finite NRE model?
3. How well do the resulting asymptotic approximations describe NRE-model toys and independent simulator-bank toys?

The second statement is an algebraic guarantee. The first and third are numerical/statistical questions: we do not force an outcome by shifting scans, recentering toy results, selecting a favorable seed, or changing the learned ratios after looking at the toys.


## Setup and persistent outputs

Run Exercise 5 first, including its simulator-toy bank cell. We read its saved PRESEL network/scaler and the cut and nominal selected yields stored in that bank. Those files are read-only; the hybrid ratio networks and reference flow are not used. If the toy metadata is unavailable, supply the exact cut and selected yields printed by Exercise 5 in the configuration below.

The checkout for this notebook is separate from the persistent Exercise 5 workspace. Set `WORK_DIR` to the directory containing `models_PRESEL/` and `simulator_toy_banks_hybrid/`. This is the legacy working directory used by the existing notebooks, not necessarily the directory containing the notebook source.

All new numerical products go to `exercise12_artifacts/`. Every displayed figure also writes a PDF and a standalone, editable Python script with embedded numerical data to `exercise12_figures_scripts/`. The scripts do not need a trained network or this notebook.

Select a **GPU runtime** for the full training and check the printed training device. Completed ensemble members, simulation banks, and toy shards are reused after a restart; an unfinished member may need to be rerun. Run only one writer per artifact directory. A retained `.lock` stops with its path rather than discarding work: remove that specific lock only after confirming that its earlier runtime has stopped.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
IN_COLAB = "google.colab" in sys.modules

# Change this only if Exercise 5 used a different persistent working directory.
WORK_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab/"
    "nsbi-lhc-toolkit/workshops/ml4hep_tifr"
) if IN_COLAB else Path.cwd()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SOURCE_REPO = Path("/content/nsbi-lhc-toolkit-exercise12")
    if not (SOURCE_REPO / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        subprocess.run(
            ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
             "--branch", BRANCH, REPO_URL, str(SOURCE_REPO)],
            check=True, env=clone_env,
        )
        subprocess.run(
            ["git", "-C", str(SOURCE_REPO), "sparse-checkout", "set",
             "workshops/ml4hep_tifr_colab"],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "-C", str(SOURCE_REPO), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )
    SOURCE_DIR = SOURCE_REPO / "workshops" / "ml4hep_tifr_colab"
    sys.path.insert(0, str(SOURCE_DIR))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "numpy", "scipy", "matplotlib", "pandas", "scikit-learn",
         "joblib", "onnxruntime", "torch"],
        check=True,
    )
else:
    SOURCE_DIR = Path.cwd()

if not WORK_DIR.is_dir():
    raise FileNotFoundError(
        f"Exercise 5 workspace not found: {WORK_DIR}. Set WORK_DIR above."
    )
ARTIFACT_ROOT = WORK_DIR / "exercise12_artifacts"
FIGURE_SCRIPT_DIR = WORK_DIR / "exercise12_figures_scripts"
print("Exercise 5 inputs (read-only):", WORK_DIR)
print("Exercise 12 artifacts:", ARTIFACT_ROOT)
print("Standalone figures:", FIGURE_SCRIPT_DIR)


In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import utils_nre
import utils_nre_inference
import utils_nre_plotting
for module in (utils_nre, utils_nre_inference, utils_nre_plotting):
    importlib.reload(module)

from utils_nre import (
    load_exercise5_selection, cached_features, cached_ratios, train_nre,
)
from utils_nre_inference import (
    finite_reference_asimov, raw_simulator_asimov, build_compression,
    simulator_bin_probabilities, run_toys_cached, validate_compression, binned_asimov,
)
from utils_nre_plotting import (
    plot_training, plot_ratio_validation, plot_mle_convergence,
    plot_asimov_scans, plot_toy_comparison, plot_compression_validation,
)

# The default is a serious training, not a quick CPU demonstration.
# SMOKE is for checking the workflow only; it cannot establish physics closure.
PROFILE = "FULL"  # "FULL" or "SMOKE"
SEED = 120_926
MU_TRUE = 1.0
PRESEL_RATIO_CUT = None
SELECTED_YIELDS = None  # Optional exact [lambda_S, lambda_B] from Exercise 5.

TRAINING_CONFIG = dict(
    ensemble_size=4,
    hidden_layers=4,
    hidden_features=1024,
    activation="swish",
    epochs=50,
    batch_size=4096,
    learning_rate=1.0e-3,
    device="auto",
    prediction_batch_size=65_536,
)
N_TRAIN_PER_CLASS = 1_000_000  # Set 5_000_000 to match Exercise 5's event budget.
N_VALIDATION_PER_CLASS = 100_000
MC_SIZES = np.array([256, 1024, 4096, 16384, 65536, 262144])
MC_REPETITIONS = 8
DEPLOYMENT_REF_EVENTS = 1_048_576
SIMULATOR_TOY_BANK_EVENTS = 1_000_000  # Per component, independent of training.
N_TOYS = 100_000
N_Q_BINS = 1024
MAX_Q_BINS = 8192  # Bounded automatic refinement before the large toy run.
N_VALIDATION_TOYS = 32
TOY_BATCH_SIZE = 128
SCAN_MU = np.linspace(0.0, 2.0, 161)

if PROFILE == "SMOKE":
    TRAINING_CONFIG.update(
        ensemble_size=2, hidden_layers=2, hidden_features=32,
        epochs=3, batch_size=256,
    )
    N_TRAIN_PER_CLASS = 4000
    N_VALIDATION_PER_CLASS = 1000
    MC_SIZES = np.array([64, 256, 1024])
    MC_REPETITIONS = 2
    DEPLOYMENT_REF_EVENTS = 4096
    SIMULATOR_TOY_BANK_EVENTS = 4096
    N_TOYS = 256
    N_Q_BINS = 64
    N_VALIDATION_TOYS = 4
elif PROFILE != "FULL":
    raise ValueError("PROFILE must be FULL or SMOKE.")
print("Profile:", PROFILE, "| training config:", TRAINING_CONFIG)


## 1. Simulator-reference NRE

Let $p_S$ and $p_B$ denote the selected simulator distributions, and keep the nominal selected yields $\lambda_S,\lambda_B$ fixed throughout the study. The reference is an **equal mixture after selection**:
$$
p_{\rm ref}(\mathbf{x})=\tfrac12p_S(\mathbf{x})+\tfrac12p_B(\mathbf{x}).
$$
Selecting events from an equal mixture *before* PRESEL would not give this distribution, because signal and background have different acceptances.

For each target $s\in\{S,B\}$, train an equally weighted binary classification problem with target label 1 and independent reference label 0. The optimum satisfies
$$
D_s(\mathbf{x})=\frac{p_s(\mathbf{x})}{p_s(\mathbf{x})+p_{\rm ref}(\mathbf{x})},
\qquad
\widehat r_s(\mathbf{x})=\frac{D_s(\mathbf{x})}{1-D_s(\mathbf{x})}.
$$
We use four independently initialized networks per component, each with four 1024-unit Swish hidden layers, as in Exercise 5, and average **ratios**, not classifier scores. Training uses BCE, NAdam, and the same learning-rate decay convention; the logits form of BCE avoids unnecessary saturation. There is no dropout, weight decay, or toy-based calibration. The default training sample is one million events per class; the configurable five-million-event setting matches Exercise 5's larger sample budget. Validation samples are independent of training.

Here “same samples” means the same simulator distributions, response, selection, and yields. We generate fresh independent draws rather than reuse a training event as an Asimov or validation event. Only reconstructed $\mathbf{x}=(x_1,\ldots,x_5)$ enters the classifiers; analytic Gaussian-mixture densities are never used in the NRE or in its normalization.


In [ ]:
selection = load_exercise5_selection(
    WORK_DIR, ratio_cut=PRESEL_RATIO_CUT, yields=SELECTED_YIELDS,
)
YIELDS = np.asarray(selection.yields, dtype=np.float64)
print("Fixed selected yields [S, B]:", YIELDS)
print("Selection fingerprint:", selection.fingerprint)

training_banks = {
    component: cached_features(
        ARTIFACT_ROOT, selection, "training", component,
        N_TRAIN_PER_CLASS, SEED + index,
    )
    for index, component in enumerate(("signal", "background", "reference"))
}
validation_banks = {
    component: cached_features(
        ARTIFACT_ROOT, selection, "validation", component,
        N_VALIDATION_PER_CLASS, SEED + 100 + index,
    )
    for index, component in enumerate(("signal", "background", "reference"))
}

nre = train_nre(
    ARTIFACT_ROOT,
    training_banks["signal"], training_banks["background"], training_banks["reference"],
    validation_banks["signal"], validation_banks["background"], validation_banks["reference"],
    TRAINING_CONFIG, SEED + 200,
)
print("Frozen NRE fingerprint:", nre.fingerprint)
plot_training(nre.histories, FIGURE_SCRIPT_DIR)
plt.show()


In [ ]:
validation_ratios = {
    name: cached_ratios(
        ARTIFACT_ROOT, selection, nre, "held_out_ratio_diagnostic", name,
        N_VALIDATION_PER_CLASS, SEED + 500 + index,
    )
    for index, name in enumerate(("signal", "background", "reference"))
}
print("Independent REF means E_REF[r_S], E_REF[r_B]:",
      validation_ratios["reference"].mean(axis=0))
plot_ratio_validation(
    validation_ratios["reference"], validation_ratios["signal"],
    validation_ratios["background"], FIGURE_SCRIPT_DIR,
)
plt.show()

# The training products are persistent. Release their in-memory copies before toys.
del training_banks, validation_banks, validation_ratios


## Fix the likelihood convention before looking at the toys

For positive ratios under the normalization convention defined below, the parameter-dependent extended log-likelihood criterion is
$$
\ell(\mu)=-(\mu\lambda_S+\lambda_B)
+\sum_i w_i\log\!\left[\mu\lambda_S r_S(\mathbf{x}_i)+\lambda_B r_B(\mathbf{x}_i)\right],
\qquad \mu\geq0.
$$
The unknown $\log p_{\rm ref}(\mathbf{x}_i)$ is independent of $\mu$ and cancels in likelihood differences. For ordinary events $w_i=1$; Asimov weights need not be integers.

We freeze one **deployment** NRE using an independent, large simulator-reference bank with
$$
c_s^{\rm deploy}=\frac1{M_{\rm deploy}}\sum_i\widehat r_s(\mathbf{x}_i),
\qquad r_s^{\rm deploy}=\widehat r_s/c_s^{\rm deploy}.
$$
All conventional simulator-Asimov fits and all external simulator-toy fits use these same constants. We never renormalize on an observed toy. This makes their comparison well defined and avoids conflating a changing likelihood with finite integration error.

The smaller reference banks below each define their own same-bank normalized finite model. Comparing their predictions to deployment-model toys is a **convergence study**; exact finite-bank closure always refers to each bank's own model.


In [ ]:
deployment_ratios = cached_ratios(
    ARTIFACT_ROOT, selection, nre, "deployment_reference", "reference",
    DEPLOYMENT_REF_EVENTS, SEED + 1000,
)
DEPLOYMENT = finite_reference_asimov(
    deployment_ratios, YIELDS, MU_TRUE, SCAN_MU,
)
DEPLOYMENT_NORMALIZERS = deployment_ratios.mean(axis=0)
print("Deployment normalizers:", DEPLOYMENT_NORMALIZERS)
print("Deployment Asimov MLE:", DEPLOYMENT["mu_hat"])


## 2. Conventional simulator-weighted Asimov samples

For $M/2$ independent selected signal events and $M/2$ background events, assign weights
$$
w_i^S=\frac{\mu_A\lambda_S}{M/2},\qquad
w_i^B=\frac{\lambda_B}{M/2},\qquad \mu_A=1.
$$
The total expected event yield is exactly $\mu_A\lambda_S+\lambda_B$, independently of $M$. Increasing $M$ improves the numerical integration; it does **not** increase the experimental luminosity.

Even for an exact likelihood, a finite random weighted dataset generally has nonzero score at $\mu_A$. Repeating this experiment lets us distinguish a random finite-$M$ displacement from a systematic shift. Monte Carlo fluctuations decrease approximately as $M^{-1/2}$ under finite-variance assumptions, but an individual sequence need not move monotonically toward 1.

With learned ratios, the large-$M$ limit is the *pseudo-true* value that maximizes the simulator-expected learned log likelihood. It equals 1 only to the extent that the NRE describes the simulator. A residual shift is a diagnostic, not something to remove by recentering the scan.


## 3. Same-reference-sample correction and exact finite-sample closure

Instead draw $M$ independent events from the simulator reference and define
$$
c_s(M)=\frac1M\sum_{i=1}^M\widehat r_s(\mathbf{x}_i),\qquad
\widetilde r_s(\mathbf{x})=\widehat r_s(\mathbf{x})/c_s(M).
$$
Use these corrected ratios both in the likelihood and in the Asimov weights,
$$
v_i(\mu)=\frac{\mu\lambda_S\widetilde r_S(\mathbf{x}_i)+
\lambda_B\widetilde r_B(\mathbf{x}_i)}{M},
\qquad w_i^A=v_i(\mu_A).
$$
Then $\sum_i v_i(\mu)=\mu\lambda_S+\lambda_B$ at every likelihood point. Consequently,
$$
\ell_A(\mu)-\ell_A(\mu_A)
=-\sum_i\left[
v_i(\mu)-v_i(\mu_A)
-v_i(\mu_A)\log\frac{v_i(\mu)}{v_i(\mu_A)}
\right]\leq0.
$$
This is a sum of nonnegative Poisson divergences with an overall minus sign. Thus $\mu_A$ is a global maximum on the finite sample, without requiring differentiability. With positive signal yield and positive ratios it is unique in this one-parameter model. In particular $t_{\mu,A}=2[\ell_A(\widehat\mu_A)-\ell_A(\mu)]\geq0$.

Nothing in this proof requires an explicit reference density or a flow. The ensemble is fixed; the finite-sample correction is only two normalization constants, not retraining. Different $M$ can still give different curvature and discovery separation even though all minima close exactly.

For each independent repetition, we generate the largest bank once and take nested prefixes. Sizes within a repetition are correlated; repetitions use distinct random streams. At each size the two constructions use the **same total number $M$ of integration events**.


In [ ]:
STUDY = {"sizes": MC_SIZES.copy(), "raw": [], "corrected": []}
for repetition in range(MC_REPETITIONS):
    stream = SEED + 10_000 + 10 * repetition
    signal_ratios = cached_ratios(
        ARTIFACT_ROOT, selection, nre, f"asimov_{repetition:02d}", "signal",
        int(MC_SIZES.max() // 2), stream,
    )
    background_ratios = cached_ratios(
        ARTIFACT_ROOT, selection, nre, f"asimov_{repetition:02d}", "background",
        int(MC_SIZES.max() // 2), stream + 1,
    )
    reference_ratios = cached_ratios(
        ARTIFACT_ROOT, selection, nre, f"asimov_{repetition:02d}", "reference",
        int(MC_SIZES.max()), stream + 2,
    )
    raw_results, corrected_results = [], []
    for size in MC_SIZES:
        n_component = int(size // 2)
        raw_results.append(raw_simulator_asimov(
            signal_ratios[:n_component], background_ratios[:n_component],
            YIELDS, MU_TRUE, SCAN_MU, normalizers=DEPLOYMENT_NORMALIZERS,
        ))
        corrected_results.append(finite_reference_asimov(
            reference_ratios[:int(size)], YIELDS, MU_TRUE, SCAN_MU,
        ))
    STUDY["raw"].append(raw_results)
    STUDY["corrected"].append(corrected_results)
    print(f"Asimov repetition {repetition + 1}/{MC_REPETITIONS} complete.")

corrected_mles = np.array([[r["mu_hat"] for r in row] for row in STUDY["corrected"]])
assert np.max(np.abs(corrected_mles - MU_TRUE)) < 1.0e-7
print("Largest corrected |MLE - 1|:", np.max(np.abs(corrected_mles - MU_TRUE)))


In [ ]:
plot_mle_convergence(STUDY, FIGURE_SCRIPT_DIR)
plot_asimov_scans(
    STUDY["raw"][0], MC_SIZES, "Simulator-weighted Asimov",
    FIGURE_SCRIPT_DIR, "simulator_asimov_scans",
)
plot_asimov_scans(
    STUDY["corrected"][0], MC_SIZES, "Same-bank corrected NRE Asimov",
    FIGURE_SCRIPT_DIR, "corrected_reference_asimov_scans",
)
plt.show()

rows = [
    dict(construction=kind, repetition=rep, M=int(size),
         mu_hat=result["mu_hat"], q0_asimov=result["q0_asimov"],
         sigma_curvature=result["sigma_curvature"])
    for kind in ("raw", "corrected")
    for rep, results in enumerate(STUDY[kind])
    for size, result in zip(MC_SIZES, results)
]
asimov_table = pd.DataFrame(rows)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
asimov_table.to_csv(ARTIFACT_ROOT / "asimov_convergence.csv", index=False)
display(asimov_table.groupby(["construction", "M"])[
    ["mu_hat", "q0_asimov", "sigma_curvature"]
].agg(["mean", "std"]))


## 4. Pseudo-experiments with a frozen NRE test statistic

There are two complementary sources:

- **NRE-model toys:** the deployment reference bank defines normalized discrete component probabilities $P_s(i)=\widehat r_s(\mathbf{x}_i)/\sum_j\widehat r_s(\mathbf{x}_j)$. Sampling these probabilities realizes exactly the finite model used in the deployment Asimov construction, without a flow.
- **Simulator-bank toys:** generate a new independent selected signal bank and background bank using the simulator, evaluate the frozen ratios once, and use their empirical probabilities. These test the simulator-to-NRE agreement, subject to finite-bank and compression accuracy.

In both cases $N_S\sim\mathrm{Poisson}(\mu_A\lambda_S)$ and $N_B\sim\mathrm{Poisson}(\lambda_B)$. We fit with the same frozen deployment likelihood, with $\widehat\mu\geq0$, and calculate
$$q_0=2[\ell(\widehat\mu)-\ell(0)],$$
with $q_0=0$ at the boundary $\widehat\mu=0$.

### Sufficient-statistic compression and its checks

Only $q(\mathbf{x})=\lambda_S r_S^{\rm deploy}(\mathbf{x})/
[\lambda_B r_B^{\rm deploy}(\mathbf{x})]$ enters the parameter-dependent likelihood,
$$\ell(\mu)-\ell(0)=-\mu\lambda_S+\sum_i\log(1+\mu q_i).$$
At the Exercise 5 yields, evaluating this sum event by event for 100,000 toys would be unnecessarily expensive. We bin this scalar using a combination of quantile and logarithmic bins, resolving both the bulk and the discovery-sensitive tails, and generate independent Poisson bin counts. The model bin probabilities come from the ratio-weighted reference bank and define the fitted bin ratios. Simulator events populate **those same bins**, but we do not replace the fitted ratios by the simulator bin-density ratios: that would silently change the NRE being tested.

The binned likelihood is a controlled approximation to the unbinned one. Before the large run, we compare binned and unbinned Asimov scans, and paired event-level and compressed fits to the same small set of toys. A full run automatically doubles the bin count up to `MAX_Q_BINS` until the checks pass. If even that is insufficient, it stops before the large toy run; increasing the limit reuses the trained models and simulation banks. A smoke run intentionally has coarse bins and poor learning; it is not a physics validation. The larger toy run resamples finite independent simulation banks, rather than calling the simulator for every event of every toy.


In [ ]:
simulator_signal = cached_ratios(
    ARTIFACT_ROOT, selection, nre, "simulator_toy_validation", "signal",
    SIMULATOR_TOY_BANK_EVENTS, SEED + 50_000,
)
simulator_background = cached_ratios(
    ARTIFACT_ROOT, selection, nre, "simulator_toy_validation", "background",
    SIMULATOR_TOY_BANK_EVENTS, SEED + 50_001,
)
# Refine compression without repeating any training or ratio evaluation.
effective_bins = int(N_Q_BINS)
if MAX_Q_BINS < effective_bins:
    raise ValueError("MAX_Q_BINS must be at least N_Q_BINS.")
while True:
    compression = build_compression(deployment_ratios, YIELDS, n_bins=effective_bins)
    simulator_probabilities = simulator_bin_probabilities(
        simulator_signal, simulator_background, compression,
    )
    binned_deployment = binned_asimov(compression, MU_TRUE, SCAN_MU)
    compression_checks = {}
    for label, extra in (
        ("model", {}),
        ("simulator", dict(signal_ratios=simulator_signal,
                          background_ratios=simulator_background)),
    ):
        validation = validate_compression(
            compression, deployment_ratios, mu_true=MU_TRUE,
            n_toys=N_VALIDATION_TOYS, seed=SEED + 60_000 + len(compression_checks),
            **extra,
        )
        compression_checks[label] = validation
        print(label, {key: value for key, value in validation.items()
                      if key.startswith(("max_abs", "rms"))})

    profile_scale = max(1.0, float(np.max(DEPLOYMENT["t_scan"])))
    profile_error = float(np.max(np.abs(
        binned_deployment["t_scan"] - DEPLOYMENT["t_scan"]
    ))) / profile_scale
    local_sigma = float(DEPLOYMENT["sigma_curvature"])
    mu_error = max(v["rms_delta_mu_hat"] for v in compression_checks.values()) / local_sigma
    q_error = max(v["rms_delta_q0"] for v in compression_checks.values()) / max(
        1.0, float(np.sqrt(DEPLOYMENT["q0_asimov"]))
    )
    print(f"Compression ({effective_bins} requested bins): relative profile error={profile_error:.3g}, "
          f"RMS MLE difference / sigma={mu_error:.3g}, "
          f"RMS q0 difference / max(1,sqrt(q0_A))={q_error:.3g}")
    # These tolerances measure numerical approximation, not statistical agreement.
    COMPRESSION_OK = profile_error < 0.005 and mu_error < 0.02 and q_error < 0.02
    if COMPRESSION_OK or PROFILE == "SMOKE":
        break
    if effective_bins >= MAX_Q_BINS:
        raise RuntimeError(
            "Compression is not yet sufficiently accurate. Increase MAX_Q_BINS and "
            "rerun from this cell; cached NN training and simulation banks are reused."
        )
    effective_bins = min(2 * effective_bins, int(MAX_Q_BINS))
    print("Refining compression to", effective_bins, "bins.")

for validation in compression_checks.values():
    plot_compression_validation(validation, FIGURE_SCRIPT_DIR)
if not COMPRESSION_OK:
    print("SMOKE only: coarse-compression check did not pass; do not interpret physics results.")
plt.show()


In [ ]:
model_toys = run_toys_cached(
    ARTIFACT_ROOT / "toys", compression,
    mu_true=MU_TRUE, n_toys=N_TOYS, seed=SEED + 70_000,
    batch_size=TOY_BATCH_SIZE,
)
simulator_toys = run_toys_cached(
    ARTIFACT_ROOT / "toys", compression,
    mu_true=MU_TRUE, n_toys=N_TOYS, seed=SEED + 70_001,
    source="simulator", probabilities=simulator_probabilities,
    batch_size=TOY_BATCH_SIZE,
)
# Completed toy batches are cached. No classifier training takes place here.
for label, toys in (("NRE model", model_toys), ("Simulator bank", simulator_toys)):
    print(label, "mean(mu_hat) =", np.mean(toys["mu_hat"]),
          "std(mu_hat) =", np.std(toys["mu_hat"], ddof=1),
          "P(q0=0) =", np.mean(toys["q0"] == 0.0))


## 5–6. Asimov asymptotic predictions versus toys

We use two familiar approximations, keeping their meanings separate:

- The local-curvature prediction is $\widehat\mu\simeq\max(0,\mathcal N(m,\sigma_{\rm curv}^2))$, where $m$ is the **actual fitted Asimov minimum**, and $\sigma_{\rm curv}^{-2}=-\ell_A''(m)$. For a corrected reference Asimov, $m=\mu_A=1$.
- The discovery approximation is $q_0\simeq[\max(0,Z+\sqrt{q_{0,A}})]^2$ for $Z\sim\mathcal N(0,1)$, using the finite-displacement Asimov separation $q_{0,A}=2[\ell_A(m)-\ell_A(0)]$.

If a conventional Asimov sample has its constrained minimum at zero while $\mu_A=1$, its usual interior-Wald overlay is omitted and flagged: the constrained fit does not identify the unconstrained Gaussian center. A spurious 50% boundary prediction would be misleading.

Both distributions contain a point mass at zero; it is included in the first displayed bin. The local curvature width and $m/\sqrt{q_{0,A}}$ need not be identical away from the strictly quadratic/Wald regime. These are asymptotic predictions, not exact finite-exposure theorems.

The first comparison uses several conventional simulator-Asimov samples from the first, preselected repetition. Their fluctuations are shown without artificially resetting their fitted minima to 1. The second uses corrected reference Asimov samples at the same sizes, plus the frozen deployment construction.

Small conventional samples can give unstable predictions. Large conventional samples can agree with simulator toys, including a possible pseudo-true displacement. Corrected samples guarantee exact internal stationarity/global closure, but their shapes still converge with $M$ and their predictions can differ from simulator toys through NRE misspecification. This is precisely why we show both toy sources instead of promising that normalization repairs every discrepancy.


In [ ]:
# Representative sizes are fixed in advance; no selection based on toy agreement.
shown = np.unique([0, len(MC_SIZES) // 2, len(MC_SIZES) - 1])
shown_labels = [f"M = {MC_SIZES[index]:,}" for index in shown]
plot_toy_comparison(
    model_toys, simulator_toys,
    [STUDY["raw"][0][index] for index in shown],
    shown_labels, "Simulator-weighted Asimov predictions",
    FIGURE_SCRIPT_DIR, "simulator_asimov_vs_nre_and_simulator_toys",
)
plot_toy_comparison(
    model_toys, simulator_toys,
    [STUDY["corrected"][0][index] for index in shown] + [DEPLOYMENT],
    shown_labels + [f"Deployment M = {DEPLOYMENT_REF_EVENTS:,}"],
    "Same-bank corrected NRE Asimov predictions",
    FIGURE_SCRIPT_DIR, "corrected_asimov_vs_nre_and_simulator_toys",
)
plt.show()


## A large-simulator expectation as a diagnostic

For completeness, use the independent simulator-toy banks themselves to estimate the expected learned likelihood at higher statistics. This estimate uses the frozen deployment normalizers, just as the simulator toys do. Its minimum need not be 1. Comparing it with the deployment-model Asimov helps identify the origin of any discrepancy:

- A small-bank raw displacement that shrinks with $M$ is Monte Carlo integration noise.
- A stable difference between the large simulator expectation and the corrected model expectation indicates learned-model or residual numerical mismatch.
- A difference between a model-consistent Asimov prediction and its own model toys, after checking compression, tests asymptotic accuracy at the fixed expected yields.

These effects are logically distinct. Same-sample correction removes the first type of internal score defect; it does not claim to solve all three.


In [ ]:
large_simulator_asimov = raw_simulator_asimov(
    simulator_signal, simulator_background, YIELDS, MU_TRUE, SCAN_MU,
    normalizers=DEPLOYMENT_NORMALIZERS,
)
plot_toy_comparison(
    model_toys, simulator_toys,
    [large_simulator_asimov, DEPLOYMENT],
    ["Large simulator expectation", "Deployment NRE-model expectation"],
    "External versus internal expected likelihood",
    FIGURE_SCRIPT_DIR, "large_simulator_vs_model_expectation",
)
plt.show()
display(pd.DataFrame([
    dict(source="large simulator", mu_hat=large_simulator_asimov["mu_hat"],
         q0_asimov=large_simulator_asimov["q0_asimov"],
         sigma_curvature=large_simulator_asimov["sigma_curvature"]),
    dict(source="deployment NRE model", mu_hat=DEPLOYMENT["mu_hat"],
         q0_asimov=DEPLOYMENT["q0_asimov"],
         sigma_curvature=DEPLOYMENT["sigma_curvature"]),
]))
print("Standalone scripts:")
for path in sorted(FIGURE_SCRIPT_DIR.glob("*.py")):
    print(path)


## What this exercise establishes

The Asimov construction is a statement about a normalized statistical model and a consistent numerical integration rule—not about a particular density estimator. A simulator-reference NRE is sufficient. For every reference-bank size we have exact finite-sample global closure, while the scan shape converges as the quadrature is refined.

There is a price for removing the flow: generating large independent reference banks now requires simulator calls. In this toy example those calls are cheap; in a realistic detector simulation they may dominate the calculation. That computational distinction is separate from the mathematical existence of the Asimov construction.

If the simulator-to-NRE comparison does not close, retain and study that result. More reference events reduce integration noise, not arbitrary classifier misspecification. Correct coverage or discovery calibration should be checked with appropriate pseudo-experiments wherever the asymptotic/model assumptions are in doubt.

Suggested variations: increase the training budget from one to five million events per class; enlarge the independent simulator-toy bank; double the compression resolution; increase the number of independent Asimov repetitions. Change one setting at a time. Cache identities keep different numerical/model configurations separate, and none of these choices alters Exercise 5 or the paper-summary trainings.
